# Packages

In [ ]:
#%pip install deepeval accelerate

  Using cached aiohttp-3.13.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached grpcio-1.78.0-cp313-cp313-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached posthog-5.4.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pytest-9.0.2-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typer-0.24.1-py3

In [ ]:
# !pip install --upgrade transformers datasets evaluate peft bitsandbytes trl

  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.2.28-cp313-cp313-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached xxhash-3.6.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 63.5 MB/s eta 0:00:00
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 39.5 MB/s eta 0:00:00
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 65.8 MB/s eta 0:00:00a 0:00:01
Using cached regex-2026.2.28-cp313-cp313-macosx_11_0_arm64.whl (289 kB)
Using cached xxhash-3.6.0-cp313-cp313-macosx_11_0_arm64.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [peft]2m11/12 [peft]formers]

[notice] A new release 

# Imports

In [4]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
import asyncio
import csv

/Users/minyeelow/Documents/GitHub/smu-llm-group-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Test Cases

In [5]:
csv_data = [
    ["input", "actual_output", "retrieval_context"],
    ["What is the capital of France?", "Paris", "Paris is the capital of France."],
    ["Who painted the Mona Lisa?", "Leonardo da Vinci painted the Mona Lisa, a famous portrait in the Louvre Museum.", "Leonardo da Vinci was an Italian polymath."],
    ["What is the tallest mountain in the world?", "Mount Everest is the Earth's highest mountain above sea level.", "Mount Everest is located in the Himalayas and is the highest mountain above sea level."]
]

with open("content.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

test_cases = []
with open("content.csv", "r") as f:
    reader = csv.reader(f)

    next(reader, None)

    for values in reader:
        if len(values) == 3:
            values = [val.strip() for val in values]

            test_cases.append(
                LLMTestCase(
                    input=values[0],
                    actual_output=values[1],
                    retrieval_context=[values[2]]
                )
            )

print(f"Loaded {len(test_cases)} test cases from file")

Loaded 3 test cases from file


# Evaluation Model

In [6]:
class EvalModel(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.pipe = None
        self.load_model(model, tokenizer)

    def load_model(self, model, tokenizer):
        self.pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
        )

        return self.pipe

    def _generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]

        outputs = self.pipe(
            messages,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.pipe.tokenizer.eos_token_id
        )

        return outputs[0]["generated_text"][-1]["content"].strip()

    # Strip kwargs at method signature level
    def generate(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        return self._generate(prompt)

    async def a_generate(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self._generate, prompt)

    def generate_raw_response(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        response = self._generate(prompt)

        return (response, 0.0)

    async def a_generate_raw_response(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        loop = asyncio.get_running_loop()

        response = await loop.run_in_executor(None, self._generate, prompt)

        return (response, 0.0)

    def get_model_name(self) -> str:
        return "Eval Model"

# Evaluate

In [7]:
'''from deepeval.metrics.g_eval.g_eval import GEval

if not hasattr(GEval, "_patched"):
    GEval._patched = True

    _original_a_evaluate = GEval._a_evaluate  # 🔥 true original

    async def safe_a_evaluate(self, test_case):
        try:
            return await _original_a_evaluate(self, test_case)

        except TypeError:
            # fix cost=None
            if self.evaluation_cost is None:
                self.evaluation_cost = 0.0
            return await _original_a_evaluate(self, test_case)

        except AttributeError:
            # fix string output issue
            res = await self.model.a_generate(self._build_prompt(test_case))
            text = res if isinstance(res, str) else str(res)
            return 5, text  # fallback

    GEval._a_evaluate = safe_a_evaluate'''

'from deepeval.metrics.g_eval.g_eval import GEval\n\nif not hasattr(GEval, "_patched"):\n    GEval._patched = True\n\n    _original_a_evaluate = GEval._a_evaluate  # 🔥 true original\n\n    async def safe_a_evaluate(self, test_case):\n        try:\n            return await _original_a_evaluate(self, test_case)\n\n        except TypeError:\n            # fix cost=None\n            if self.evaluation_cost is None:\n                self.evaluation_cost = 0.0\n            return await _original_a_evaluate(self, test_case)\n\n        except AttributeError:\n            # fix string output issue\n            res = await self.model.a_generate(self._build_prompt(test_case))\n            text = res if isinstance(res, str) else str(res)\n            return 5, text  # fallback\n\n    GEval._a_evaluate = safe_a_evaluate'

In [8]:
'''from deepeval.metrics.g_eval.g_eval import GEval

original_eval = GEval._a_evaluate

async def safe_a_evaluate(self, test_case):
    try:
        return await original_eval(self, test_case)

    except TypeError:
        # Fix cost=None
        if self.evaluation_cost is None:
            self.evaluation_cost = 0.0
        return await original_eval(self, test_case)

    except AttributeError:
        # 🔥 Fix: model returns string instead of ReasonScore
        res = await self.model.a_generate(self._build_prompt(test_case))

        # fallback parsing
        text = res if isinstance(res, str) else str(res)

        # simple heuristic score
        if "10" in text:
            score = 10
        elif "9" in text:
            score = 9
        elif "8" in text:
            score = 8
        elif "7" in text:
            score = 7
        else:
            score = 5

        return score, text

GEval._a_evaluate = safe_a_evaluate'''

'from deepeval.metrics.g_eval.g_eval import GEval\n\noriginal_eval = GEval._a_evaluate\n\nasync def safe_a_evaluate(self, test_case):\n    try:\n        return await original_eval(self, test_case)\n\n    except TypeError:\n        # Fix cost=None\n        if self.evaluation_cost is None:\n            self.evaluation_cost = 0.0\n        return await original_eval(self, test_case)\n\n    except AttributeError:\n        # 🔥 Fix: model returns string instead of ReasonScore\n        res = await self.model.a_generate(self._build_prompt(test_case))\n\n        # fallback parsing\n        text = res if isinstance(res, str) else str(res)\n\n        # simple heuristic score\n        if "10" in text:\n            score = 10\n        elif "9" in text:\n            score = 9\n        elif "8" in text:\n            score = 8\n        elif "7" in text:\n            score = 7\n        else:\n            score = 5\n\n        return score, text\n\nGEval._a_evaluate = safe_a_evaluate'

In [9]:
'''print("Evaluating")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

evaluator = EvalModel(model, tokenizer)

# ✅ DEFINE metrics FIRST
factual_metric = GEval(
    name="Factuality & Grounding",
    criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",
    evaluation_steps=[
        "List all factual claims made in the actual_output.",
        "Verify each claim is directly supported by retrieval_context.",
        "Penalize hallucinations, speculation, or unsupported details.",
        "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT
    ],
    model=evaluator,
    threshold=0.7
)

recall_metric = GEval(
    name="Recall",
    criteria="Assess if retrieval_context contains all information needed to fully answer the input query.",
    evaluation_steps=[
        "Identify key facts required to answer the input query.",
        "Check if retrieval_context includes these key facts.",
        "Penalize missing critical information needed for complete answer.",
        "Score 1-10: 10=context fully supports ideal answer, 1=critical info missing."
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT
    ],
    model=evaluator,
    threshold=0.7
)

# 🔥 IMPORTANT FIX
factual_metric.evaluation_cost = 0.0
recall_metric.evaluation_cost = 0.0

# ✅ RUN
results = evaluate(
    test_cases,
    [factual_metric, recall_metric]
)'''

'print("Evaluating")\n\nmodel = AutoModelForCausalLM.from_pretrained(\n    "microsoft/Phi-3.5-mini-instruct",\n    torch_dtype=torch.float16,\n    trust_remote_code=True,\n).to("cuda")\n\ntokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")\n\nevaluator = EvalModel(model, tokenizer)\n\n# ✅ DEFINE metrics FIRST\nfactual_metric = GEval(\n    name="Factuality & Grounding",\n    criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",\n    evaluation_steps=[\n        "List all factual claims made in the actual_output.",\n        "Verify each claim is directly supported by retrieval_context.",\n        "Penalize hallucinations, speculation, or unsupported details.",\n        "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."\n    ],\n    evaluation_params=[\n        LLMTestCaseParams.ACTUAL_OUTPUT,\n        LLMTestCaseParams.RETRIEVAL_CONTEXT\n    ],\n    model=evaluator,\n    threshold=0.7\n)\n\nrecall_metr

In [10]:
print("Evaluating")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

evaluator = EvalModel(model, tokenizer)

results = evaluate(
    test_cases,
    [
      GEval(
          name="Factuality & Grounding",
          criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",
          evaluation_steps=[
              "List all factual claims made in the actual_output.",
              "Verify each claim is directly supported by retrieval_context.",
              "Penalize hallucinations, speculation, or unsupported details.",
              "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."
          ],
          evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
          model=evaluator,
          threshold=0.7
      ),
      GEval(
          name="Recall",
          criteria="Assess if retrieval_context contains all information needed to fully answer the input query.",
          evaluation_steps=[
              "Identify key facts required to answer the input query.",
              "Check if retrieval_context includes these key facts.",
              "Penalize missing critical information needed for complete answer.",
              "Score 1-10: 10=context fully supports ideal answer, 1=critical info missing."
          ],
          evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
          model=evaluator,
          threshold=0.7
      )
    ]
)

Evaluating


A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`flash

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
print("\n=== RESULTS ===")
print(f"{'Case':<4} {'Fact':<6} {'Rec':<6} {'Pass':<6} {'Reason':<50}")
print("-" * 90)

for idx, result in enumerate(results.test_results):
  fact_score = result.metrics_data[0].score  # Factuality
  recall_score = result.metrics_data[1].score  # Recall
  fact_reason = result.metrics_data[0].reason[:45]

  status = "Y" if fact_score >= 0.7 and recall_score >= 0.7 else "N"

  print(f"{idx:<4} {fact_score:<6.3f} {recall_score:<6.3f} {status:<6} {fact_reason}")

fact_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7) / len(results.test_results)
recall_pass = sum(1 for r in results.test_results if r.metrics_data[1].score >= 0.7) / len(results.test_results)
both_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7 and r.metrics_data[1].score >= 0.7) / len(results.test_results)

print(f"\nSummary:")
print(f"Factuality Pass Rate: {fact_pass:.1%}")
print(f"Recall Pass Rate: {recall_pass:.1%}")
print(f"Both Pass Rate: {both_pass:.1%}")


=== RESULTS ===
Case Fact   Rec    Pass   Reason                                            
------------------------------------------------------------------------------------------


NameError: name 'results' is not defined